# Espagne (ES) — réplication du modèle de prix

Version légère : **modèle de prix seul** (pas de valorisation sur actifs), même spécification que la France, formes exponentielle et LSTAR.

**Spécificités de l'Espagne**
- Les premiers prix négatifs sont apparus au printemps 2024 ; auparavant, le surplus se traduisait par des prix nuls ou quasi nuls. L'événement étudié est donc **prix ≤ 0** (`event_cut = 0.01`), et l'échantillon s'étend jusqu'en 2025 (test = 2025).
- Le marché journalier est passé au pas de 15 minutes le 1er octobre 2025 : les prix sont moyennés à l'heure.
- La panne générale du 28 avril 2025 crée des observations atypiques dans le test.
- Interconnexion limitée (France, Portugal) : x est dominé par la production solaire et la charge.

**Corrections par rapport au notebook d'origine**

1. **Fuite d'information supprimée** : les instruments XGBoost n'utilisent plus que des retards physiques de 48 h et plus (connus à la clôture J-1 12h), et sont entraînés uniquement sur l'échantillon d'estimation (plus sur la validation).
2. **Contraintes correctement imposées** : bornes dans l'optimiseur au lieu de `clip`/`max` dans le modèle (avant : α₂₄ estimé à −0,175 mais utilisé à 0,01, et seuils calculés sur des θ non contraints). Jacobien analytique et plusieurs points de départ.
3. **Paramètre `phi_ar` supprimé** : il n'était jamais utilisé, donc non identifié.
4. **Bootstrap cohérent** : bootstrap par paires en blocs de 7 jours, au lieu d'une simulation récursive qui utilisait le prix de l'heure précédente alors que le modèle utilise celui de 23h la veille.
5. **Calibration de l'incertitude corrigée** : la variable `xt_val` n'est plus écrasée (avant, la calibration portait sur une seule observation, ce qui donnait σ ≈ 1,01 au lieu de 0,65). Les probabilités d'événement utilisent la distribution empirique des résidus de validation.
6. **Benchmarks équitables** : l'ARX a les mêmes 4 retards que le modèle ; le HMM est filtré (probabilités d'état à la fin de J-1) au lieu d'être simulé au hasard.
7. **Diagnostics de cohérence des seuils** : position des seuils dans la distribution de x, fréquence des prix négatifs de part et d'autre, point de bascule empirique (logit) exprimé en % de la charge, comparable entre marchés.
8. **Option LSTAR** : transition logistique bornée G(x) = 1/(1+exp(−γ(x−c))), dont le paramètre c est un vrai seuil de localisation.
9. **Trous de données** : `regularize` comble les trous <= 3 h par interpolation et les trous de 4 à 48 h par la valeur de la même heure sept jours plus tôt ; les lignes reconstruites sont marquées `imputed` et **exclues de l'estimation, de la validation et du test** (aucun prix reconstruit n'entre dans une mesure de performance). `gap_policy="drop"` les ignore à la place.
10. **Instrumentation** : solde net (imports - exports) instrumenté d'un bloc, profils horaires et moyennes mobiles sur 7 jours, jours fériés, prévision J-1 éolien+solaire en option ; le R² de x lui-même est affiché.
11. **Bootstrap** : correction d'un `KeyError` (clés datetime) par des codes entiers de jours, et parallélisation (`n_jobs`).
12. **Divers** : `freq='h'` (pandas récent), gestion des changements d'heure (calcul en UTC), boucles vectorisées.

## 1. Paramètres

In [ ]:
import os
ENTSOE_TOKEN = os.environ.get("ENTSOE_TOKEN", "COLLEZ_VOTRE_JETON_ICI")
SOURCE = "entsoe"                 # "entsoe" (recommandé) ou "energy-charts" (prix ES peut-être indisponible)
ZONE = "ES"
YEARS = (2019, 2025)
HYDRO_RESERVOIR_IN_R = False
PARAMS = dict(market="ES", tz="Europe/Madrid",
              val_start="2024-07-01", test_start="2025-01-01", test_end="2026-01-01",
              event_cut=0.01,             # événement = prix <= 0 (voir note)
              leak_free=True, load_is_forecast=(SOURCE == "entsoe"), holiday_country="ES", B=199)

## 2. Fonctions du modèle

In [ ]:
# Installation (une seule fois)
# !pip install numpy pandas scipy scikit-learn xgboost hmmlearn matplotlib entsoe-py requests holidays joblib

In [ ]:
# ============================================================================
# FONCTIONS DU MODÈLE (identiques dans les quatre notebooks)
# ============================================================================
import numpy as np, pandas as pd, json, os, warnings
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from scipy.optimize import least_squares
from scipy.stats import norm
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score, brier_score_loss
warnings.filterwarnings("ignore")

LAGS = ["p_last", "p_24", "p_48", "p_168"]   # p_last = dernier prix du jour J-1 (23h locale), connu à la clôture
COMPONENTS = ["Intercept", "Lag J-1 23h", "Lag 24h", "Lag 48h", "Lag 168h"]


@dataclass
class Config:
    market: str = "FR"
    tz: str = "Europe/Paris"
    val_start: str = "2023-07-01"
    test_start: str = "2024-01-01"
    test_end: str = "2025-01-01"
    event_cut: float = 0.0          # événement = prix < event_cut (0 : prix strictement négatif)
    leak_free: bool = True          # retards physiques >= 48 h (disponibles à la clôture J-1 12h)
    load_is_forecast: bool = True   # charge = prévision J-1 -> non instrumentée
    price_cap: float = 1000.0       # pics > cap remplacés par la médiane mobile 24 h (None = désactivé)
    theta_bounded: tuple = (1, 2)   # |theta| < 0.99 pour les retards J-1 23h et 24h (spécification du papier)
    alpha_bounds: tuple = (1e-3, 10.0)
    n_starts: int = 4
    B: int = 199                    # répliques bootstrap
    block_days: int = 7             # bootstrap par blocs de 7 jours
    seed: int = 42
    gap_policy: str = "weekly_naive"   # trous 4-48 h : "weekly_naive" (même heure J-7, lignes marquées et exclues) ou "drop"
    max_interp: int = 3                # trous <= 3 h : interpolation linéaire (lignes marquées et exclues)
    max_fill: int = 48                 # taille maximale d'un trou comblé
    net_trade: bool = True             # instrumente le solde net (imports - exports) au lieu de deux séries creuses
    holiday_country: str | None = None # code pays du package `holidays` ("FR", "SE", "DE", "ES")
    n_jobs: int = -1                   # parallélisme du bootstrap (-1 = tous les coeurs)


# ---------------------------------------------------------------- préparation des données
def localize_naive_local(index, tz):
    """Horodatage local naïf (heure d'hiver dupliquée) -> UTC sans perte d'observation."""
    amb = ~pd.Index(index).duplicated(keep="first")
    return index.tz_localize(tz, ambiguous=np.asarray(amb), nonexistent="shift_forward").tz_convert("UTC")


def regularize(df, tz, max_interp=3, max_fill=48, policy="weekly_naive"):
    """Grille horaire UTC continue.
    - trous <= max_interp h : interpolation linéaire ;
    - trous <= max_fill h (policy="weekly_naive") : valeur de la même heure 7 jours plus tôt ;
    - plus longs : laissés vides (lignes écartées).
    Toute ligne qui avait au moins une valeur manquante est marquée imputed=True : elle sert à
    conserver la structure des retards, mais est EXCLUE de l'estimation, de la validation et du test."""
    df = df[~df.index.duplicated(keep="first")].sort_index()
    full = pd.date_range(df.index.min(), df.index.max(), freq="h", tz="UTC")
    df = df.reindex(full)
    cols = list(df.columns)
    missing = df[cols].isna().any(axis=1)
    df[cols] = df[cols].interpolate(limit=max_interp, limit_area="inside")
    if policy == "weekly_naive":
        for c in cols:
            na = df[c].isna()
            if not na.any():
                continue
            run_len = na.groupby((na != na.shift()).cumsum()).transform("sum")
            fillable = na & (run_len <= max_fill)
            df.loc[fillable, c] = df[c].shift(168)[fillable]
    df["imputed"] = missing.values
    df["local"] = df.index.tz_convert(tz)
    n_gap = int(missing.sum())
    print(f"Grille horaire : {len(df):,} heures, {n_gap} heures reconstruites (exclues de l'estimation et de l'évaluation), "
          f"{int(df[cols].isna().any(axis=1).sum())} encore vides")
    return df


def add_features(df, cfg):
    df = df.copy()
    if cfg.price_cap is not None:
        spikes = df["price"] > cfg.price_cap
        med = df["price"].rolling(24, center=True, min_periods=6).median()
        df.loc[spikes, "price"] = med[spikes]
        print(f"Pics > {cfg.price_cap} €/MWh remplacés : {int(spikes.sum())}")
    loc = df["local"]
    df["hour"], df["dow"] = loc.dt.hour, loc.dt.dayofweek
    df["weekend"] = (df["dow"] >= 5).astype(int)
    doy = loc.dt.dayofyear
    df["is_holiday"], df["near_holiday"] = 0, 0
    if cfg.holiday_country:
        try:
            import holidays as _hol
            years = range(int(loc.dt.year.min()) - 1, int(loc.dt.year.max()) + 2)
            hset = pd.to_datetime(list(_hol.country_holidays(cfg.holiday_country, years=years).keys()))
            dloc = loc.dt.normalize().dt.tz_localize(None)
            df["is_holiday"] = dloc.isin(hset).astype(int).values
            df["near_holiday"] = (dloc.isin(hset + pd.Timedelta(days=1)) | dloc.isin(hset - pd.Timedelta(days=1))).astype(int).values
        except ImportError:
            print("Package `holidays` absent : jours fériés ignorés (pip install holidays)")
    df["sin_h"], df["cos_h"] = np.sin(2*np.pi*df.hour/24), np.cos(2*np.pi*df.hour/24)
    df["sin_d"], df["cos_d"] = np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)
    day = loc.dt.normalize().dt.tz_localize(None)
    last = df.groupby(day)["price"].last()
    df["p_last"] = (day - pd.Timedelta(days=1)).map(last).values          # vectorisé
    for k in (24, 48, 168):
        df[f"p_{k}"] = df["price"].shift(k)
    return df


def instrument(df, cfg):
    """Instrumentation XGBoost entraînée UNIQUEMENT sur l'échantillon d'estimation (lignes non reconstruites).
    - Si une prévision J-1 éolien+solaire ('vre_forecast') existe, elle sert de variable explicative connue ;
      si 'renewable_other' existe aussi, seul ce reliquat est instrumenté (x = vre_forecast + reliquat).
    - Le solde net (imports - exports) est instrumenté d'un bloc (cfg.net_trade) : x ne dépend que de I - E."""
    df = df.copy()
    has_vre = "vre_forecast" in df and df["vre_forecast"].notna().mean() > 0.9
    use_fc = has_vre and "renewable_other" in df
    ren = "renewable_other" if use_fc else "renewable"
    df["net_trade"] = df["imports"] - df["exports"]
    trade = ["net_trade"] if cfg.net_trade else ["imports", "exports"]
    endog = [ren, "nuclear"] + trade + ["storage"] + ([] if cfg.load_is_forecast else ["load"])
    known = (["load"] if cfg.load_is_forecast else []) + (["vre_forecast"] if has_vre else [])
    cal = ["sin_h", "cos_h", "sin_d", "cos_d", "dow", "weekend", "is_holiday", "near_holiday"]
    lags = (48, 72, 168) if cfg.leak_free else (1, 24, 48, 168)
    base = lags[0]
    lagvars = list(dict.fromkeys(endog + known + ["imports", "exports"]))
    feats = cal + known
    for v in lagvars:
        for k in lags:
            df[f"{v}_l{k}"] = df[v].shift(k); feats.append(f"{v}_l{k}")
        # profil horaire moyen des 7 derniers jours et moyenne glissante 7 jours (info disponible à t - base)
        df[f"{v}_h7"] = pd.concat([df[v].shift(base + 24 * k) for k in range(7)], axis=1).mean(axis=1)
        df[f"{v}_m7"] = df[v].shift(base).rolling(168, min_periods=96).mean()
        feats += [f"{v}_h7", f"{v}_m7"]
    ok = df[feats].notna().all(axis=1)
    fit_rows = ok & ~df["imputed"]
    tr = fit_rows & (df.index < pd.Timestamp(cfg.val_start, tz="UTC"))
    te = fit_rows & (df.index >= pd.Timestamp(cfg.test_start, tz="UTC"))
    r2 = {}
    for v in endog:
        if df[v].fillna(0).abs().sum() == 0:
            df[f"{v}_hat"] = 0.0; continue
        m = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.03, subsample=0.8,
                         colsample_bytree=0.8, reg_lambda=2.0, min_child_weight=5,
                         random_state=cfg.seed, n_jobs=-1)
        fit_mask = tr & df[v].notna()
        m.fit(df.loc[fit_mask, feats], df.loc[fit_mask, v])
        df.loc[ok, f"{v}_hat"] = m.predict(df.loc[ok, feats])
        yt = df.loc[te, v].dropna()
        r2[v] = round(float(1 - np.mean((yt - df.loc[yt.index, f"{v}_hat"])**2) / np.var(yt)), 3) if len(yt) else np.nan
    if cfg.load_is_forecast:
        df["load_hat"] = df["load"]
    R_hat = (df["vre_forecast"] + df["renewable_other_hat"]) if use_fc else df["renewable_hat"]
    T_hat = df["net_trade_hat"] if cfg.net_trade else (df["imports_hat"] - df["exports_hat"])
    df["x_raw"] = R_hat + df.nuclear_hat + T_hat - df.load_hat - df.storage_hat
    # diagnostic principal : qualité de x lui-même (ce qui compte pour le modèle), pas des composantes
    R_act = (df["vre_forecast"] + df["renewable_other"]) if use_fc else df["renewable"]
    x_act = R_act + df.nuclear + df["net_trade"] - df.load - df.storage
    xt = pd.concat([x_act[te], df.x_raw[te]], axis=1).dropna()
    r2_x = float(1 - ((xt.iloc[:, 0] - xt.iloc[:, 1])**2).mean() / xt.iloc[:, 0].var()) if len(xt) else np.nan
    print(f"Prévision J-1 éolien+solaire utilisée : {has_vre} (reliquat instrumenté : {use_fc}) | solde net instrumenté : {cfg.net_trade}")
    print("R² des instruments sur le test :", r2)
    print(f"R² de x lui-même (x prévu vs x réalisé, test) : {r2_x:.3f}")
    df.attrs["r2_instruments_test"] = r2; df.attrs["r2_x_test"] = r2_x
    return df


# ---------------------------------------------------------------- forme exponentielle (papier)
def f_exp(p, x, Z):
    P = p.reshape(5, 3)
    return np.sum((P[:, 0] + P[:, 1] * np.exp(np.outer(x, P[:, 2]))) * Z, axis=1)

def jac_exp(p, x, Z):
    P = p.reshape(5, 3); E = np.exp(np.outer(x, P[:, 2]))
    J = np.empty((len(x), 15))
    J[:, 0::3], J[:, 1::3], J[:, 2::3] = Z, E * Z, P[:, 1] * x[:, None] * E * Z
    return -J

def fit_exp(x, Z, y, cfg, p0=None):
    lo, hi = np.full(15, -np.inf), np.full(15, np.inf)
    lo[2::3], hi[2::3] = cfg.alpha_bounds
    for k in cfg.theta_bounded:
        lo[3*k+1], hi[3*k+1] = -0.99, 0.99
    starts = [p0] if p0 is not None else []
    if p0 is None:   # initialisation par linéarisation exp(a x) ~ 1 + a x
        beta = np.linalg.lstsq(np.column_stack([Z, Z * x[:, None]]), y, rcond=None)[0]
        for a0 in np.linspace(0.3, 1.5, cfg.n_starts):
            p = np.zeros(15)
            p[0::3], p[1::3], p[2::3] = beta[:5] - beta[5:] / a0, beta[5:] / a0, a0
            starts.append(p)
    best = None
    for s in starts:
        r = least_squares(lambda p: y - f_exp(p, x, Z), np.clip(s, lo + 1e-6, hi - 1e-6),
                          jac=lambda p: jac_exp(p, x, Z), bounds=(lo, hi), method="trf",
                          x_scale="jac", max_nfev=2000)
        if best is None or r.cost < best.cost: best = r
    best.lo, best.hi = lo, hi
    return best

def thresholds_exp(p, med, iqr, x_tr, load_mean, lo, hi):
    P, rows = p.reshape(5, 3), []
    for k, name in enumerate(COMPONENTS):
        th, al = P[k, 1], P[k, 2]
        tau = -np.log(abs(th)) / al if th != 0 else np.nan
        mwh = med + iqr * tau
        at_b = bool(np.isclose(al, lo[3*k+2], rtol=1e-3) or np.isclose(al, hi[3*k+2], rtol=1e-3)
                    or (np.isfinite(lo[3*k+1]) and abs(abs(th) - 0.99) < 1e-3))
        pct = float((x_tr < mwh).mean() * 100)
        rows.append(dict(component=name, phi=P[k, 0], theta=th, alpha=al, tau_MWh=mwh,
                         tau_pct_load=100 * mwh / load_mean, percentile_x_train=pct,
                         at_bound=at_b, inside_support=0.5 < pct < 99.5))
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- forme logistique (LSTAR, bornée)
def _G(x, g, c): return 1.0 / (1.0 + np.exp(-g * (x - c)))

def f_lstar(p, x, Z):
    return Z @ p[:5] + _G(x, p[10], p[11]) * (Z @ p[5:10])

def jac_lstar(p, x, Z):
    G = _G(x, p[10], p[11]); zt = Z @ p[5:10]; dG = G * (1 - G)
    return -np.column_stack([Z, Z * G[:, None], dG * (x - p[11]) * zt, -dG * p[10] * zt])

def fit_lstar(x, Z, y, cfg, p0=None):
    lo, hi = np.full(12, -np.inf), np.full(12, np.inf)
    lo[10], hi[10] = 0.1, 50.0
    lo[11], hi[11] = np.percentile(x, 0.5), np.percentile(x, 99.9)
    starts = [p0] if p0 is not None else []
    if p0 is None:
        phi = np.linalg.lstsq(Z, y, rcond=None)[0]
        starts += [np.r_[phi, np.zeros(5), g0, c0] for c0 in np.percentile(x, [80, 90, 97, 99.5]) for g0 in (2.0, 8.0)]
    best = None
    for s in starts:
        r = least_squares(lambda p: y - f_lstar(p, x, Z), np.clip(s, lo + 1e-9, hi - 1e-9),
                          jac=lambda p: jac_lstar(p, x, Z), bounds=(lo, hi), method="trf",
                          x_scale="jac", max_nfev=2000)
        if best is None or r.cost < best.cost: best = r
    best.lo, best.hi = lo, hi
    return best

def thresholds_lstar(p, med, iqr, x_tr, load_mean, lo, hi):
    g, c = p[10], p[11]; mwh = med + iqr * c
    pct = float((x_tr < mwh).mean() * 100)
    at_b = bool(np.isclose(c, lo[11], rtol=1e-4) or np.isclose(c, hi[11], rtol=1e-4) or np.isclose(g, hi[10]))
    return pd.DataFrame([dict(component="Transition commune c", gamma=g, tau_MWh=mwh,
                              tau_pct_load=100 * mwh / load_mean, percentile_x_train=pct,
                              transition_width_MWh=iqr * np.log(81) / g,   # G de 10 % à 90 %
                              at_bound=at_b, inside_support=0.5 < pct < 99.5)])


# ---------------------------------------------------------------- bootstrap par blocs de jours
def _boot_fit(fitter, x, Z, y, sel, cfg, p_hat):
    return fitter(x[sel], Z[sel], y[sel], cfg, p0=p_hat).x


def block_bootstrap(x, Z, y, days, cfg, p_hat, fitter):
    """Bootstrap par paires, en blocs de cfg.block_days jours consécutifs.
    Les jours sont repérés par des codes entiers (pas de clés datetime : source de KeyError)."""
    codes, _ = pd.factorize(pd.Series(np.asarray(days)))
    cuts = np.flatnonzero(np.diff(codes)) + 1
    groups = np.split(np.arange(len(codes)), cuts)            # une entrée par jour, ordre chronologique
    n_days, bl = len(groups), cfg.block_days
    nb = int(np.ceil(n_days / bl))
    rng = np.random.default_rng(cfg.seed)
    sels = []
    for _ in range(cfg.B):
        st = rng.integers(0, n_days - bl + 1, nb)
        sels.append(np.concatenate([np.concatenate(groups[s:s + bl]) for s in st]))
    if cfg.n_jobs == 1:
        out = []
        for b, sel in enumerate(sels):
            out.append(_boot_fit(fitter, x, Z, y, sel, cfg, p_hat))
            if (b + 1) % 10 == 0: print(f"   bootstrap {b+1}/{cfg.B}")
    else:
        from joblib import Parallel, delayed
        print(f"   bootstrap : {cfg.B} répliques en parallèle (n_jobs={cfg.n_jobs})")
        out = Parallel(n_jobs=cfg.n_jobs, verbose=5)(delayed(_boot_fit)(fitter, x, Z, y, sel, cfg, p_hat) for sel in sels)
    return np.array(out)


# ---------------------------------------------------------------- benchmark HMM filtré (équitable)
def hmm_filtered(resid, fit_mask, day_codes, seed=42):
    """HMM gaussien à 2 états sur les résidus ARX. Prévision J-1 : probabilités filtrées à la dernière
    heure de J-1, propagées k pas par la matrice de transition (aucune information du jour J)."""
    from hmmlearn.hmm import GaussianHMM
    m = GaussianHMM(n_components=2, covariance_type="diag", n_iter=300, random_state=seed)
    m.fit(resid[fit_mask][:, None])
    A, mu = m.transmat_, m.means_[:, 0]
    sd = np.sqrt(np.asarray(m.covars_).reshape(2, -1)[:, 0])
    lik = norm.pdf(resid[:, None], mu, sd) + 1e-300
    alpha = np.empty_like(lik); a = m.startprob_.copy()
    for t in range(len(resid)):
        a = (a @ A if t else a) * lik[t]; a /= a.sum(); alpha[t] = a
    pos = np.arange(len(resid))
    last_prev = pd.Series(pos).groupby(day_codes).transform("min").values - 1
    k = np.clip(pos - last_prev, 1, 30)
    Ak = {j: np.linalg.matrix_power(A, j) for j in np.unique(k)}
    probs = np.array([alpha[max(lp, 0)] @ Ak[kk] for lp, kk in zip(last_prev, k)])
    return probs @ mu


# ---------------------------------------------------------------- pipeline complet
def run_model(df_std, cfg, form="exponential", bootstrap=True, prepared=None):
    """form = 'exponential' (spécification du papier) ou 'logistic' (LSTAR)."""
    if prepared is None:
        d = instrument(add_features(regularize(df_std, cfg.tz, cfg.max_interp, cfg.max_fill, cfg.gap_policy), cfg), cfg)
        d = d.dropna(subset=["price", "x_raw"] + LAGS)
        d = d[d.index < pd.Timestamp(cfg.test_end, tz="UTC")]
    else:
        d = prepared.dropna(subset=["price", "x_raw"] + LAGS)
        d = d[d.index < pd.Timestamp(cfg.test_end, tz="UTC")]
    vs, ts = pd.Timestamp(cfg.val_start, tz="UTC"), pd.Timestamp(cfg.test_start, tz="UTC")
    valid = ~d["imputed"].values
    tr, va, te = (d.index < vs) & valid, (d.index >= vs) & (d.index < ts) & valid, (d.index >= ts) & valid
    ev = d.price.values < cfg.event_cut
    print(f"\n===== {cfg.market} | forme = {form} =====")
    print(f"Observations train/val/test : {tr.sum()}/{va.sum()}/{te.sum()} | événements (prix < {cfg.event_cut}) : "
          f"{ev[tr].sum()}/{ev[va].sum()}/{ev[te].sum()}")
    n_ev = int(ev[tr].sum()); n_tot = int(tr.sum())
    if n_ev == n_tot:
        raise ValueError(f"Tous les {n_tot} prix de l'estimation sont < {cfg.event_cut} €/MWh. "
                         "Vérifiez la base de données ou augmentez event_cut.")
    if n_ev < 20:
        print(f"⚠️  Seulement {n_ev} événements dans l'estimation sur {n_tot} observations.")
    if ev[tr].sum() < 50:
        print("⚠️ Moins de 50 événements dans l'échantillon d'estimation : la non-linéarité sera mal identifiée.")

    xr = d.x_raw.values
    med = np.median(xr[tr]); iqr = np.subtract(*np.percentile(xr[tr], [75, 25])); x = (xr - med) / iqr
    a = np.median(d.price[tr]); b = 1.4826 * np.median(np.abs(d.price[tr] - a))
    T = lambda v: np.arcsinh((v - a) / b); Tinv = lambda u: np.sinh(u) * b + a
    y = T(d.price.values)
    assert d[LAGS + ["price", "x_raw"]].notna().all().all(), "NaN résiduels dans les régresseurs après dropna — vérifier la base de données"
    Z = np.column_stack([np.ones(len(d))] + [T(d[c].values) for c in LAGS])
    load_mean = d.load[tr].mean()

    fitter, F = (fit_lstar, f_lstar) if form == "logistic" else (fit_exp, f_exp)
    fit = fitter(x[tr], Z[tr], y[tr], cfg); p = fit.x
    thr = (thresholds_lstar if form == "logistic" else thresholds_exp)(p, med, iqr, xr[tr], load_mean, fit.lo, fit.hi)

    boot = None
    if bootstrap and cfg.B > 0:
        days = d["local"][tr].dt.normalize().values
        boot = block_bootstrap(x[tr], Z[tr], y[tr], days, cfg, p, fitter)
        if form == "logistic":
            thr["CI_low_MWh"], thr["CI_high_MWh"] = np.percentile(med + iqr * boot[:, 11], [2.5, 97.5])
        else:
            taus = np.array([[-np.log(abs(bp.reshape(5, 3)[k, 1])) / bp.reshape(5, 3)[k, 2] for k in range(5)] for bp in boot])
            ci = np.nanpercentile(med + iqr * taus, [2.5, 97.5], axis=0)
            thr["CI_low_MWh"], thr["CI_high_MWh"] = ci[0], ci[1]
            thr["se_theta"], thr["se_alpha"] = boot[:, 1::3].std(0, ddof=1), boot[:, 2::3].std(0, ddof=1)

    # prévisions J-1 : tous les régresseurs sont connus à la clôture
    f = F(p, x, Z)
    e_val = np.sort(y[va] - f[va])                                  # résidus hors échantillon (validation)
    prob = np.searchsorted(e_val, T(cfg.event_cut) - f, side="left") / len(e_val)
    pred = pd.DataFrame({"price": d.price.values, "x_MWh": xr, "STAR_median": Tinv(f),
                         "STAR_mean": np.array([Tinv(fi + e_val).mean() for fi in f]),
                         "prob_event": prob}, index=d.index)

    # benchmarks avec exactement les mêmes régresseurs (x + 4 retards)
    Xb = np.column_stack([np.ones(len(y)), x, Z[:, 1:]])
    beta = np.linalg.lstsq(Xb[tr], y[tr], rcond=None)[0]
    arx = Xb @ beta; pred["ARX"] = Tinv(arx)
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=cfg.seed)
    rf.fit(Xb[tr, 1:], y[tr]); pred["RF"] = Tinv(rf.predict(Xb[:, 1:]))
    try:
        codes = pd.factorize(d["local"].dt.normalize())[0]
        pred["HMM"] = Tinv(arx + hmm_filtered(y - arx, tr, codes, cfg.seed))
    except Exception as e:
        print("HMM indisponible :", e)

    rows = []
    for lab, m in (("validation", va), ("test", te)):
        o = pred.price.values[m]; evm = o < cfg.event_cut
        for col in [c for c in ["STAR_median", "STAR_mean", "ARX", "RF", "HMM"] if c in pred]:
            pr = pred[col].values[m]
            sc = pred.prob_event.values[m] if col.startswith("STAR") else -pr
            rows.append(dict(sample=lab, model=col, RMSE=np.sqrt(mean_squared_error(o, pr)),
                             MAE=mean_absolute_error(o, pr),
                             RMSE_events=np.sqrt(mean_squared_error(o[evm], pr[evm])) if evm.any() else np.nan,
                             AUC_events=roc_auc_score(evm, sc) if 0 < evm.sum() < len(evm) else np.nan,
                             Brier=brier_score_loss(evm, pred.prob_event.values[m]) if col.startswith("STAR") else np.nan))
    metrics = pd.DataFrame(rows)

    # cohérence économique des seuils
    coh = []
    for _, r in thr.iterrows():
        for lab, m in (("train", tr), ("test", te)):
            above = xr[m] > r.tau_MWh; e = ev[m]
            coh.append(dict(component=r.component, sample=lab, tau_MWh=r.tau_MWh, hours_above=int(above.sum()),
                            P_event_above=e[above].mean() if above.any() else np.nan,
                            P_event_below=e[~above].mean(),
                            share_events_above=e[above].sum() / max(e.sum(), 1)))
    coherence = pd.DataFrame(coh)

    # point de bascule empirique (indépendant de la normalisation -> comparable entre marchés)
    tip = []
    for lab, m in (("train", tr), ("test", te)):
        e = ev[m]
        if e.sum() >= 10:
            lr = LogisticRegression(C=1e6).fit(xr[m][:, None] / 1000, e)
            b0, b1 = lr.intercept_[0], lr.coef_[0, 0]
            x10, x50 = 1000 * (np.log(1/9) - b0) / b1, -1000 * b0 / b1
        else:
            x10 = x50 = np.nan
        tip.append(dict(sample=lab, events=int(e.sum()), x_median_events=np.median(xr[m][e]) if e.any() else np.nan,
                        x_p99=np.percentile(xr[m], 99), x_max=xr[m].max(), tipping_P10_MWh=x10,
                        tipping_P50_MWh=x50, tipping_P50_pct_load=100 * x50 / d.load[m].mean()))
    tipping = pd.DataFrame(tip)

    return dict(cfg=cfg, form=form, params=p, thr=thr, boot=boot, metrics=metrics, coherence=coherence,
                tipping=tipping, pred=pred, prepared=d, norm=dict(med=med, iqr=iqr, a=a, b=b),
                masks=dict(tr=tr, va=va, te=te))


def show(res):
    from IPython.display import display
    print(f"\n--- Seuils ({res['form']}) ---"); display(res["thr"].round(3))
    print("--- Performances ---"); display(res["metrics"].round(3))
    print("--- Cohérence des seuils ---"); display(res["coherence"].round(4))
    print("--- Point de bascule empirique ---"); display(res["tipping"].round(1))


def plot_diagnostics(res, fname=None):
    d, p, cfg, nm = res["prepared"], res["params"], res["cfg"], res["norm"]
    tr = res["masks"]["tr"]; xr = d.x_raw.values
    grid = np.linspace(np.percentile(xr, 0.1), xr.max(), 300); xg = (grid - nm["med"]) / nm["iqr"]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    if res["form"] == "logistic":
        ax[0].plot(grid, _G(xg, p[10], p[11]), lw=2, label="G(x) : poids du régime de surplus")
    else:
        P = p.reshape(5, 3)
        for k, n in enumerate(COMPONENTS):
            ax[0].plot(grid, P[k, 0] + P[k, 1] * np.exp(P[k, 2] * xg), label=n)
        ax[0].set_ylim(-3, 3)
    for t in res["thr"].tau_MWh: ax[0].axvline(t, ls="--", c="grey", alpha=.6)
    ax[0].axvline(np.percentile(xr[tr], 99), c="red", lw=1, label="99e percentile de x (estimation)")
    ax[0].set_xlabel("x (MWh)"); ax[0].set_title(f"{cfg.market} – coefficients en fonction de x"); ax[0].legend(fontsize=8)
    bins = pd.qcut(xr, 20, duplicates="drop")
    rate = pd.Series(d.price.values < cfg.event_cut).groupby(bins, observed=True).mean()
    ax[1].bar(range(len(rate)), rate.values)
    ax[1].set_xticks(range(len(rate))); ax[1].set_xticklabels([f"{iv.right:,.0f}" for iv in rate.index], rotation=90, fontsize=7)
    ax[1].set_xlabel("Borne supérieure du vingtile de x (MWh)"); ax[1].set_ylabel("Fréquence des événements")
    ax[1].set_title("Fréquence des prix négatifs par vingtile de x (tout l'échantillon)")
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()


def export_summary(results, market):
    out = {"market": market, "models": {}}
    for form, r in results.items():
        mt = r["metrics"]; t = mt[mt["sample"] == "test"].set_index("model")
        out["models"][form] = dict(
            thresholds=r["thr"][["component", "tau_MWh", "tau_pct_load", "percentile_x_train", "at_bound", "inside_support"]].to_dict("records"),
            tipping=r["tipping"].to_dict("records"),
            test_metrics=t[["RMSE", "MAE", "RMSE_events", "AUC_events"]].round(3).to_dict("index"))
    with open(f"summary_{market}.json", "w") as fh:
        json.dump(out, fh, indent=2, default=float)
    print(f"✓ summary_{market}.json écrit")


def compare_markets():
    import glob
    rows = []
    for f in sorted(glob.glob("summary_*.json")):
        s = json.load(open(f))
        for form, m in s["models"].items():
            tp = {r["sample"]: r for r in m["tipping"]}
            first = m["thresholds"][0]
            rows.append(dict(market=s["market"], form=form, threshold_MWh=first["tau_MWh"],
                             threshold_pct_load=first["tau_pct_load"], percentile_x=first["percentile_x_train"],
                             inside_support=first["inside_support"],
                             tipping_P50_pct_load_train=tp.get("train", {}).get("tipping_P50_pct_load"),
                             events_train=tp.get("train", {}).get("events"),
                             RMSE_test_STAR=m["test_metrics"].get("STAR_mean", {}).get("RMSE"),
                             RMSE_test_ARX=m["test_metrics"].get("ARX", {}).get("RMSE")))
    return pd.DataFrame(rows)


## 3. Téléchargement des données (ENTSO-E)

In [ ]:
# ============================================================================
# TÉLÉCHARGEMENT ENTSO-E (jeton gratuit : https://transparency.entsoe.eu, "Web API Security Token")
# Les données brutes sont mises en cache dans un CSV : relancer le notebook ne retélécharge rien.
# ============================================================================
# Patch pour entsoe-py >= 0.6.11 sur Windows (UnknownTimeZoneError: Europe/Amsterdam)
import os, sys
try:
    import pytz; pytz.timezone("Europe/Amsterdam")
except Exception:
    try:
        from zoneinfo import ZoneInfo; ZoneInfo("Europe/Amsterdam")
    except Exception:
        # Forcer l'installation du package tzdata manquant
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "tzdata", "--quiet"], check=False)
# Patch monkey-patch au cas où la version installée de entsoe-py est cassée
try:
    import pandas as _pd
    import entsoe.entsoe as _ee
    if hasattr(_ee, 'QUARTER_MTU_SDAC_GOLIVE'):
        _ee.QUARTER_MTU_SDAC_GOLIVE = _pd.Timestamp('2025-10-01', tz='Europe/Paris')
except Exception:
    pass

from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

RENEW_OTHER = ["Hydro Run-of-river and poundage", "Biomass", "Geothermal", "Other renewable", "Marine"]
VRE = ["Wind Onshore", "Wind Offshore", "Solar"]

def _hourly(obj):
    if isinstance(obj, pd.DataFrame) and obj.shape[1] == 1:
        obj = obj.iloc[:, 0]
    obj.index = pd.DatetimeIndex(obj.index).tz_convert("UTC")
    return obj.resample("h").mean()                    # 15 min -> horaire (prix 15 min depuis oct. 2025)

def _query(fn, label, zone, start, end, **kw):
    """Requête annuelle, puis mensuelle en cas d'échec (limites de l'API)."""
    try:
        return fn(zone, start=start, end=end, **kw)
    except NoMatchingDataError:
        print(f"   {label} : aucune donnée"); return None
    except Exception as e:
        print(f"   {label} : échec annuel ({type(e).__name__}), reprise mois par mois")
        parts = []
        for ms in pd.date_range(start, end, freq="MS", inclusive="left"):
            me = min(ms + pd.offsets.MonthBegin(1), end)
            try:
                parts.append(fn(zone, start=ms, end=me, **kw))
            except Exception as e2:
                print(f"      {label} {ms:%Y-%m} : {type(e2).__name__}")
        return pd.concat(parts) if parts else None

def _split_generation(g):
    """query_generation renvoie soit des colonnes simples, soit (type, Actual Aggregated/Consumption)."""
    if isinstance(g.columns, pd.MultiIndex):
        prod = g.xs("Actual Aggregated", axis=1, level=1, drop_level=True) if "Actual Aggregated" in g.columns.get_level_values(1) else pd.DataFrame(index=g.index)
        cons = g.xs("Actual Consumption", axis=1, level=1, drop_level=True) if "Actual Consumption" in g.columns.get_level_values(1) else pd.DataFrame(index=g.index)
    else:
        prod, cons = g, pd.DataFrame(index=g.index)
    return prod, cons

def fetch_entsoe(zone, y0, y1, token, hydro_reservoir_in_R=False, cache=None):
    cache = cache or f"data_{zone}.csv"
    if os.path.exists(cache):
        print(f"✓ Cache trouvé : {cache}")
        df = pd.read_csv(cache, index_col=0); df.index = pd.to_datetime(df.index, utc=True); return df
    client = EntsoePandasClient(api_key=token)
    frames = []
    for yr in range(y0, y1 + 1):
        s = pd.Timestamp(f"{yr}-01-01", tz="Europe/Brussels"); e = pd.Timestamp(f"{yr+1}-01-01", tz="Europe/Brussels")
        print(f"→ {zone} {yr}")
        cols = {}
        pr = _query(client.query_day_ahead_prices, "prix", zone, s, e)
        if pr is not None: cols["price"] = _hourly(pr)
        lf = _query(client.query_load_forecast, "prévision de charge", zone, s, e)
        if lf is not None: cols["load"] = _hourly(lf)
        ws = _query(client.query_wind_and_solar_forecast, "prévision éolien+solaire", zone, s, e)
        if ws is not None:
            ws.index = pd.DatetimeIndex(ws.index).tz_convert("UTC")
            cols["vre_forecast"] = ws.resample("h").mean().sum(axis=1, min_count=1)
        g = _query(client.query_generation, "production par filière", zone, s, e)
        if g is not None:
            prod, cons = _split_generation(g)
            prod.index = pd.DatetimeIndex(prod.index).tz_convert("UTC"); prod = prod.resample("h").mean()
            if yr == y0: print("   filières disponibles :", list(prod.columns))
            other = RENEW_OTHER + (["Hydro Water Reservoir"] if hydro_reservoir_in_R else [])
            vre = prod[[c for c in VRE if c in prod]].sum(axis=1, min_count=1)
            oth = prod[[c for c in other if c in prod]].sum(axis=1, min_count=1).fillna(0)
            cols["renewable"] = vre + oth
            cols["renewable_other"] = oth
            cols["nuclear"] = prod["Nuclear"] if "Nuclear" in prod else pd.Series(0.0, index=prod.index)
            if "Hydro Pumped Storage" in cons:
                cons.index = pd.DatetimeIndex(cons.index).tz_convert("UTC")
                cols["storage"] = cons["Hydro Pumped Storage"].resample("h").mean()
        for direction, name in ((False, "imports"), (True, "exports")):
            fl = _query(client.query_physical_crossborder_allborders, f"flux ({name})", zone, s, e, export=direction)
            if fl is not None:
                fl.index = pd.DatetimeIndex(fl.index).tz_convert("UTC")
                tot = fl["sum"] if "sum" in fl else fl.sum(axis=1)
                cols[name] = tot.resample("h").mean()
        frames.append(pd.DataFrame(cols))
    df = pd.concat(frames)
    df = df[~df.index.duplicated()].sort_index()
    for c in ["nuclear", "storage", "imports", "exports", "renewable_other"]:
        if c not in df: df[c] = 0.0
        df[c] = df[c].fillna(0.0)
    df.to_csv(cache)
    print(f"✓ Données enregistrées dans {cache}")
    return df


### Alternative sans jeton : Energy-Charts

In [ ]:
# ============================================================================
# ALTERNATIVE : TÉLÉCHARGEMENT ENERGY-CHARTS (Fraunhofer ISE, CC BY 4.0, sans jeton)
# Charge RÉALISÉE seulement -> elle est instrumentée (LOAD_IS_FORECAST = False).
# Vérifiez au premier lancement la correspondance des filières imprimée.
# ============================================================================
import requests, time

EC_API = "https://api.energy-charts.info"
EC_MAPPING = [("pumped storage consumption", "storage"), ("cross border", None), ("residual load", None),
              ("renewable share", None), ("load", "load"), ("nuclear", "nuclear"), ("wind", "renewable"),
              ("solar", "renewable"), ("run-of-river", "renewable"), ("biomass", "renewable"),
              ("geothermal", "renewable"), ("water reservoir", None), ("hydro", None)]
CBPF_SIGN = +1   # +1 si valeur > 0 = import ; à vérifier avec check_flows()

def _ec_get(endpoint, **params):
    for attempt in range(5):
        r = requests.get(f"{EC_API}/{endpoint}", params=params, timeout=120)
        if r.status_code == 200: return r.json()
        time.sleep(5 * (attempt + 1))
    r.raise_for_status()

def _ec_series(unix, values):
    s = pd.Series([np.nan if v is None else v for v in values], index=pd.to_datetime(unix, unit="s", utc=True), dtype="float64")
    return s.resample("h").mean()

def fetch_energy_charts(country, bzn, y0, y1, cache=None):
    cache = cache or f"data_EC_{country.upper()}.csv"
    if os.path.exists(cache):
        print(f"✓ Cache trouvé : {cache}")
        df = pd.read_csv(cache, index_col=0); df.index = pd.to_datetime(df.index, utc=True); return df
    frames = []
    for yr in range(y0, y1 + 1):
        print(f"→ {country.upper()} {yr}")
        rng = dict(start=f"{yr}-01-01", end=f"{yr}-12-31T23:59")
        j = _ec_get("public_power", country=country, **rng)
        cols = {}
        for pt in j["production_types"]:
            tgt = next((t for k, t in EC_MAPPING if k in pt["name"].lower()), "NON ASSOCIÉ")
            if yr == y0: print(f"   {pt['name']:<45} -> {tgt}")
            if tgt in (None, "NON ASSOCIÉ"): continue
            s = _ec_series(j["unix_seconds"], pt["data"])
            if tgt == "storage": s = s.abs()
            cols[tgt] = s if tgt not in cols else cols[tgt].add(s, fill_value=0)
        fl = _ec_get("cbpf", country=country, **rng)
        idx = pd.to_datetime(fl["unix_seconds"], unit="s", utc=True)
        imp = pd.Series(0.0, index=idx); exp = pd.Series(0.0, index=idx)
        for c in fl["countries"]:
            if c["name"].lower() in ("sum", "total"): continue
            v = CBPF_SIGN * pd.Series([np.nan if z is None else z for z in c["data"]], index=idx).fillna(0)
            imp += v.clip(lower=0); exp += (-v).clip(lower=0)
        cols["imports"], cols["exports"] = imp.resample("h").mean(), exp.resample("h").mean()
        pj = _ec_get("price", bzn=bzn, **rng)
        cols["price"] = _ec_series(pj["unix_seconds"], pj["price"])
        frames.append(pd.DataFrame(cols))
    df = pd.concat(frames); df = df[~df.index.duplicated()].sort_index()
    for c in ["nuclear", "storage", "imports", "exports"]:
        if c not in df: df[c] = 0.0
        df[c] = df[c].fillna(0.0)
    df.to_csv(cache); print(f"✓ Données enregistrées dans {cache}")
    return df

def check_flows(df):
    """Contrôle du signe des flux : midi estival = généralement exportateur pour DE et ES."""
    m = df.index.month.isin([6, 7]) & df.index.hour.isin([10, 11])
    net = (df.imports - df.exports)[m].mean()
    print(f"Solde net moyen midi estival : {net:,.0f} MW -> {'exportateur (signe plausible)' if net < 0 else 'importateur : vérifier CBPF_SIGN'}")


## 4. Données au format standard et contrôles

In [ ]:
cfg = Config(**PARAMS)
if SOURCE == "entsoe":
    df_std = fetch_entsoe(ZONE, *YEARS, token=ENTSOE_TOKEN, hydro_reservoir_in_R=HYDRO_RESERVOIR_IN_R, cache="data_ES.csv")
else:
    df_std = fetch_energy_charts("es", "ES", *YEARS)
    check_flows(df_std)
print(df_std.describe().T.round(1))
print("Taux de couverture par variable :"); print(df_std.notna().mean().round(3))
print("Prix <= 0 par an :"); print((df_std.price <= 0).groupby(df_std.index.year).sum())

## 4 bis. Audit des données (avant toute estimation)

Le verdict `BLOQUANT` interrompt le notebook à la cellule suivante ; `ATTENTION` signale des points à documenter dans l'article.

In [ ]:
# ============================================================================
# AUDIT DES DONNÉES — à passer AVANT d'appliquer le modèle
# Renvoie un rapport (tableau) et un verdict : OK / ATTENTION / BLOQUANT.
# ============================================================================
def audit_data(df, cfg, expect_vre_forecast=False, plot=True):
    from IPython.display import display
    issues = []                      # (gravité, message)
    def flag(level, msg): issues.append((level, msg))
    d = df.copy()
    if d.index.tz is None:
        d.index = d.index.tz_localize("UTC")
    req = ["price", "load", "nuclear", "renewable", "imports", "exports", "storage"]
    miss = [c for c in req if c not in d.columns]
    if miss: flag("BLOQUANT", f"colonnes absentes : {miss}")
    d = d[[c for c in req + ["vre_forecast", "renewable_other"] if c in d.columns]]

    # 1. structure temporelle
    dup = int(d.index.duplicated().sum())
    d = d[~d.index.duplicated(keep="first")].sort_index()
    full = pd.date_range(d.index.min(), d.index.max(), freq="h", tz="UTC")
    holes = full.difference(d.index)
    print(f"Période : {d.index.min():%Y-%m-%d} -> {d.index.max():%Y-%m-%d} | {len(d):,} heures présentes sur {len(full):,}")
    if dup: flag("ATTENTION", f"{dup} horodatages dupliqués (le premier est conservé)")
    if len(holes):
        sh = pd.Series(holes); runs = sh.diff().ne(pd.Timedelta("1h")).cumsum()
        gaps = sh.groupby(runs.values).agg(debut="min", fin="max", heures="size")
        mi, mf = getattr(cfg, "max_interp", 3), getattr(cfg, "max_fill", 48)
        n_i, n_f, n_x = int(gaps.heures[gaps.heures <= mi].sum()), int(gaps.heures[(gaps.heures > mi) & (gaps.heures <= mf)].sum()), int(gaps.heures[gaps.heures > mf].sum())
        print(f"{len(holes)} heures absentes en {len(gaps)} trous : {n_i} h dans des trous <= {mi} h (interpolation), "
              f"{n_f} h dans des trous de {mi+1} à {mf} h (même heure J-7), {n_x} h dans des trous plus longs (non comblés)")
        long_gaps = gaps[gaps.heures > mi]
        if len(long_gaps):
            print(f"Trous de plus de {mi} h :"); display(long_gaps)
        if n_x:
            flag("BLOQUANT" if gaps.heures.max() > 168 else "ATTENTION", f"{n_x} h dans des trous > {mf} h : lignes écartées, retards perdus ensuite")
        flag("ATTENTION", f"{n_i + n_f} heures reconstruites : exclues de l'estimation et de l'évaluation (voir regularize)")

    # 2. couverture par variable et par année
    cov = d.notna().groupby(d.index.year).mean().T.round(3)
    print("\nTaux de couverture (part des heures non vides) par année :"); display(cov)
    for c in d.columns:
        tot = d[c].notna().mean()
        if tot < 0.95 and c not in ("vre_forecast",):
            flag("BLOQUANT" if c in ("price", "load") else "ATTENTION", f"{c} : couverture globale {tot:.1%}")
    if expect_vre_forecast and ("vre_forecast" not in d or d["vre_forecast"].notna().mean() < 0.9):
        flag("ATTENTION", "prévision éolien+solaire J-1 absente ou < 90 % : le modèle instrumentera toute la production renouvelable (R² plus faible)")

    # 3. plausibilité physique
    pr = d.price
    print(f"\nPrix : min {pr.min():.1f} | médiane {pr.median():.1f} | max {pr.max():.1f} €/MWh")
    if pr.min() < -500 or pr.max() > 4000: flag("ATTENTION", f"prix hors plage plausible ({pr.min():.0f} ; {pr.max():.0f})")
    for c in ["nuclear", "renewable", "imports", "exports", "storage"]:
        if c in d and (d[c] < -1e-6).mean() > 0.001:
            flag("ATTENTION", f"{c} : {(d[c] < 0).mean():.1%} de valeurs négatives (convention de signe à vérifier)")
    if "load" in d and (d.load <= 0).any(): flag("ATTENTION", f"charge <= 0 : {(d.load <= 0).sum()} heures")
    for c in ["price", "load", "renewable"]:
        if c in d:
            run = d[c].ne(d[c].shift()).cumsum()
            stuck = d[c].groupby(run).transform("size")
            n_stuck = int(((stuck >= 12) & d[c].notna() & (d[c].abs() > 1e-9)).sum())
            if n_stuck: flag("ATTENTION", f"{c} : {n_stuck} heures dans des plateaux constants >= 12 h (valeur figée / donnée recopiée ?)")
    if "storage" in d and d.storage.fillna(0).abs().sum() == 0: flag("ATTENTION", "storage = 0 partout (pompage absent)")
    if "nuclear" in d and d.nuclear.fillna(0).abs().sum() == 0: print("   (nucléaire = 0 partout : normal pour SE2 ; x ne contient pas ce terme)")

    # 4. cohérence : bilan transfrontalier et heure d'été
    if "imports" in d and "exports" in d:
        m = d.index.month.isin([6, 7]) & (d.index.tz_convert(cfg.tz).hour.isin([12, 13]))
        net = (d.imports - d.exports)[m].mean()
        print(f"Solde net moyen aux heures de midi en été : {net:,.0f} MW (négatif = exportateur net)")
    loc = d.index.tz_convert(cfg.tz)
    n_dst = (pd.Series(loc.hour, index=d.index).groupby(loc.normalize()).size().value_counts())
    print("Heures par jour local (23 = passage à l'heure d'été, 25 = retour à l'heure d'hiver ; autres valeurs = jours incomplets) :", {int(k): int(v) for k, v in n_dst.items()})

    # 5. valeurs aberrantes (écart robuste à la médiane mobile)
    for c in ["price", "load", "renewable"]:
        if c in d:
            med = d[c].rolling(168, center=True, min_periods=48).median()
            mad = (d[c] - med).abs().rolling(168, center=True, min_periods=48).median() + 1e-9
            out = int((((d[c] - med).abs() / (1.4826 * mad)) > 12).sum())
            print(f"   {c} : {out} valeurs à > 12 écarts robustes de la médiane mobile (hors prix, à inspecter)" if c != "price"
                  else f"   price : {out} pics extrêmes ({(pr > cfg.price_cap).sum()} au-dessus de {cfg.price_cap:.0f} €/MWh seront lissés par le modèle)")

    # 6. événements par année et fenêtres d'estimation
    ev = (pr < cfg.event_cut)
    ev_year = ev.groupby(ev.index.year).sum()
    print(f"\nÉvénements (prix < {cfg.event_cut}) par année :"); print(ev_year.to_dict())
    vs, ts = pd.Timestamp(cfg.val_start, tz="UTC"), pd.Timestamp(cfg.test_start, tz="UTC")
    n_tr, n_va, n_te = int(ev[d.index < vs].sum()), int(ev[(d.index >= vs) & (d.index < ts)].sum()), int(ev[d.index >= ts].sum())
    print(f"Événements : estimation {n_tr} | validation {n_va} | test {n_te}")
    if n_tr < 50: flag("BLOQUANT" if n_tr < 20 else "ATTENTION", f"seulement {n_tr} événements dans l'estimation : non-linéarité mal identifiée (avancer val_start ?)")
    if n_te < 30: flag("ATTENTION", f"seulement {n_te} événements dans le test : AUC et RMSE sur événements peu fiables")
    if n_va < 10: flag("ATTENTION", f"seulement {n_va} événements en validation : calibration des probabilités fragile")

    # 7. stabilité du régime de marché (rupture de niveau de prix)
    ym = pr.groupby(pr.index.year).agg(["mean", "std"]).round(1)
    print("\nPrix moyen / écart-type par année (repérer 2021-2022 et les changements de zone de prix) :"); display(ym.T)

    if plot:
        fig, ax = plt.subplots(2, 2, figsize=(15, 8))
        d.notna().resample("MS").mean()[[c for c in req if c in d]].plot(ax=ax[0, 0], lw=1); ax[0, 0].set_title("Couverture mensuelle"); ax[0, 0].set_ylim(0, 1.05)
        pr.resample("W").mean().plot(ax=ax[0, 1], lw=.8); ax[0, 1].set_title("Prix hebdomadaire moyen (€/MWh)")
        ev.groupby(ev.index.to_period("Q")).sum().plot(kind="bar", ax=ax[1, 0]); ax[1, 0].set_title("Événements par trimestre")
        for lab in ax[1, 0].get_xticklabels(): lab.set_fontsize(6)
        ax[1, 1].hist(pr.clip(-100, 300), bins=120); ax[1, 1].set_title("Distribution des prix (tronquée à -100 ; 300)")
        plt.tight_layout(); plt.show()

    rank = {"BLOQUANT": 2, "ATTENTION": 1}
    worst = max([rank[l] for l, _ in issues], default=0)
    verdict = ["OK", "ATTENTION", "BLOQUANT"][worst]
    print("\n" + "=" * 70); print(f"VERDICT DE L'AUDIT : {verdict}"); print("=" * 70)
    for l, m in sorted(issues, key=lambda t: -rank[t[0]]): print(f"  [{l}] {m}")
    if not issues: print("  Aucun problème détecté.")
    if verdict == "BLOQUANT":
        print("\n⛔ Corrigez ou complétez les données avant de lancer le modèle (voir messages BLOQUANT).")
    return verdict, issues


In [ ]:
verdict, issues = audit_data(df_std, cfg, expect_vre_forecast=(PARAMS.get("load_is_forecast", False)))
assert verdict != 'BLOQUANT', 'Audit bloquant : corriger les donnees avant de continuer'

## 5. Instrumentation et construction de x

In [ ]:
prepared = instrument(add_features(regularize(df_std, cfg.tz, cfg.max_interp, cfg.max_fill, cfg.gap_policy), cfg), cfg)
prepared = prepared.dropna(subset=["price", "x_raw"] + LAGS)
prepared = prepared[prepared.index < pd.Timestamp(cfg.test_end, tz="UTC")]

## 6. Forme exponentielle (spécification du papier)

In [ ]:
RES = {}
RES["exponential"] = run_model(None, cfg, "exponential", bootstrap=cfg.B > 0, prepared=prepared)
show(RES["exponential"]); plot_diagnostics(RES["exponential"], f"{cfg.market}_diagnostics_exponential.png")

## 7. Forme logistique (LSTAR)

In [ ]:
RES["logistic"] = run_model(None, cfg, "logistic", bootstrap=cfg.B > 0, prepared=prepared)
show(RES["logistic"]); plot_diagnostics(RES["logistic"], f"{cfg.market}_diagnostics_logistic.png")

## 8. Les seuils ont-ils une interprétation cohérente ?

Critères à vérifier, dans cet ordre :
1. **Signes** : forme exponentielle θ₀ < 0 et α₀ > 0 (le surplus tire le prix vers le bas) ; LSTAR : le coefficient de l'intercept en régime de surplus (φ₀ + θ₀) est inférieur à φ₀.
2. **Seuil dans le support** : `inside_support = True` et `at_bound = False`. Un seuil au-delà du 99,5e percentile de x est une extrapolation.
3. **Saut de fréquence** : `P_event_above` nettement supérieur à `P_event_below`, sur l'estimation **et** sur le test.
4. **Part des événements expliquée** : `share_events_above` ; si elle est faible, la plupart des prix négatifs surviennent sous le seuil.
5. **Comparabilité** : comparer les seuils et le point de bascule en **% de la charge moyenne**, pas en MWh.

In [ ]:
for form, r in RES.items():
    p = r["params"]
    if form == "exponential":
        sign_ok = p[1] < 0 and p[2] > 0
    else:
        sign_ok = p[5] < 0
    t = r["thr"].iloc[0]; c = r["coherence"]; c0 = c[c.component == t.component].set_index("sample")
    print(f"{form:12s} | signes {'OK' if sign_ok else 'NON'} | seuil {t.tau_MWh:,.0f} MWh ({t.tau_pct_load:.1f} % charge, "
          f"percentile {t.percentile_x_train:.1f}) | dans le support : {t.inside_support} | borne : {t.at_bound}")
    for s in ("train", "test"):
        if s in c0.index:
            print(f"   {s:5s} : P(év.|x>seuil) = {c0.loc[s,'P_event_above']:.3f} vs {c0.loc[s,'P_event_below']:.3f} sous le seuil ; "
                  f"part des événements au-dessus = {c0.loc[s,'share_events_above']:.1%}")

## 9. Export et comparaison avec la France

In [ ]:
export_summary(RES, cfg.market)
display(compare_markets())